# WP11: high-tail triad phase and amplitude audit

Run the cells in order in Google Colab. This is a short finite-Galerkin simulation, not a proof of global Navier–Stokes regularity. The source code and mathematical identities live in draft PR #27.


In [ ]:
!git clone --quiet --branch wp11-one-sided-cross-frequency-20260925 https://github.com/reggaesharkk/navier-stokes-bridge-audit.git /content/wp11-repo
!python3 /content/wp11-repo/src/wp11_phase_rate_audit.py --output /content/wp11_phase.json


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

report = json.loads(Path("/content/wp11_phase.json").read_text())
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for branch in report["results"]:
    t = [row["time"] for row in branch["rows"]]
    label = f"N={branch['cutoff']}"
    axes[0].plot(t, [row["high_transfer"] for row in branch["rows"]], label=label)
    axes[1].plot(t, [row["covariance_cos_phase_amplitude_growth"] for row in branch["rows"]], label=f"{label}: amplitude covariance")
    axes[1].plot(t, [row["weighted_mean_sin_phase_rate"] for row in branch["rows"]], linestyle="--", label=f"{label}: signed phase term")
    print(label, "integrated signed transfer:", branch["integrated_high_transfer"])
    print("worst identity residual:", max(row["covariance_identity_error"] for row in branch["rows"]))
axes[0].axhline(0, color="black", linewidth=0.7)
axes[0].set_ylabel("Signed high-advector transfer")
axes[1].set_ylabel("Terms in the exact phase identity (1/time)")
axes[1].set_xlabel("Time")
for axis in axes:
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)
fig.tight_layout()
plt.show()


The dashed curves are the weighted mean of `sin(phi) * phi_dot`. They enter the exact transfer derivative with a **minus** sign and must be combined with amplitude growth and the active-mass derivative. The plotted covariance is between `cos(phi)` and relative amplitude growth. Thresholded or zero triads retain a separate direct derivative. Changes between N=4 and N=7 are sample observations; they say nothing by themselves about the N→∞ bound. See `notes/WP11_PHASE_RATE_COVARIANCE_GATE_2026_09_25.md`.


In [ ]:
from google.colab import files
files.download("/content/wp11_phase.json")
